# S03 — Define Exposure, Endpoints, and Analysis Cohorts

This notebook:
1) loads processed data from S2 (`pcos_qc.parquet` preferred),
2) defines thyroid autoimmunity exposure variables:
   - `tai_A`, `tai_B`, `tai_C` (pre-specified)
   - optional `tai_D`, `tai_E` if enabled in config
3) defines endpoints:
   - primary: TG/HDL > cutoff
   - secondary: non-HDL >= cutoff
   - secondary: OGTT 120' glucose >= cutoff
4) defines euthyroid flags:
   - `euthyroid_tsh_only`
   - `euthyroid_strict` (optional; requires FT4 ref range in config)
5) defines explicit analysis-eligibility flags for downstream models
6) exports:
   - analysis-ready dataset (`pcos_analysis.parquet`, `pcos_analysis.csv`)
   - counts tables for exposures, endpoints, and cohort flow

Important:
- S3 does NOT fit models.
- S3 does NOT apply trimming or winsorization-based exclusions.
- S3 defines variables and analysis cohorts explicitly for downstream notebooks.

## Imports

In [ ]:
import json
import logging
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd

## Config

In [ ]:
def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def resolve_config_path() -> Path:
    candidates = [
        Path("/content/reports/config_snapshot.json"),
        Path("/mnt/data/config_snapshot.json"),
        Path("/content/config.json"),
        Path("/mnt/data/config.json"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "No config found. Expected config_snapshot.json or config.json in /content or /mnt/data."
    )

CONFIG_PATH = resolve_config_path()
CFG = load_json(CONFIG_PATH)

print("Loaded config:", str(CONFIG_PATH))

Loaded config: /content/config.json


## Directories and logging

In [ ]:
def ensure_dirs(cfg: dict) -> None:
    path_keys = [
        "output_dir",
        "intermediate_dir",
        "figures_dir",
        "tables_dir",
        "models_dir",
        "reports_dir",
        "qc_dir",
        "supplementary_dir",
    ]
    for key in path_keys:
        if key in cfg.get("paths", {}):
            Path(cfg["paths"][key]).mkdir(parents=True, exist_ok=True)

def setup_logging(cfg: dict) -> None:
    if not cfg.get("logging", {}).get("enabled", False):
        return

    root_logger = logging.getLogger()
    if root_logger.handlers:
        root_logger.setLevel(logging.INFO)
        return

    log_file = Path(cfg["logging"]["log_file"])
    log_file.parent.mkdir(parents=True, exist_ok=True)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.FileHandler(log_file, encoding="utf-8"),
            logging.StreamHandler(),
        ],
    )
    logging.info("Logging initialized in S3.")
    logging.info("Config loaded from: %s", str(CONFIG_PATH))

ensure_dirs(CFG)
setup_logging(CFG)

paths = CFG["paths"]
intermediate_dir = Path(paths["intermediate_dir"])
tables_dir = Path(paths["tables_dir"])
reports_dir = Path(paths["reports_dir"])
qc_dir = Path(paths["qc_dir"])

## Load pcos_qc

In [ ]:
def resolve_qc_data_paths(cfg: dict) -> List[Path]:
    candidates = [
        Path(cfg["paths"]["intermediate_dir"]) / "pcos_qc.parquet",
        Path(cfg["paths"]["intermediate_dir"]) / "pcos_qc.csv",
        Path("/content/pcos_qc.parquet"),
        Path("/content/pcos_qc.csv"),
        Path("/mnt/data/pcos_qc.parquet"),
        Path("/mnt/data/pcos_qc.csv"),
    ]
    return candidates

def load_qc_dataset(cfg: dict) -> Tuple[pd.DataFrame, str]:
    candidates = resolve_qc_data_paths(cfg)

    parquet_candidates = [p for p in candidates if p.suffix == ".parquet" and p.exists()]
    csv_candidates = [p for p in candidates if p.suffix == ".csv" and p.exists()]

    for p in parquet_candidates:
        try:
            df = pd.read_parquet(p)
            return df, str(p)
        except Exception as e:
            logging.warning("Failed to read parquet %s: %r", str(p), e)

    for p in csv_candidates:
        try:
            df = pd.read_csv(p)
            return df, str(p)
        except Exception as e:
            logging.warning("Failed to read csv %s: %r", str(p), e)

    raise FileNotFoundError(
        "Could not locate pcos_qc.parquet or pcos_qc.csv in configured/intermediate locations."
    )

df, source_used = load_qc_dataset(CFG)

print("Loaded:", source_used)
print("Shape:", df.shape)
df.head(3)

Loaded: /content/pcos_qc.parquet
Shape: (1300, 59)


,id,age,anti_tpo,anti_tg,tsh,ft4,ft3,tg,hdl,tc,...,flag_glu120_negative,tg_hdl_ratio,non_hdl,out_tg_hdl_ratio,out_non_hdl,flag_non_hdl_negative,log1p_anti_tpo,log1p_tg,log1p_ins0,log1p_ins120
0,7611,25.0,13.8,NaN,0.969,1.20,NaN,116.0,56.6,188.0,...,False,2.049470,131.4,False,False,False,2.694627,4.762174,NaN,NaN
1,8133,25.0,12.6,NaN,2.050,1.18,NaN,144.0,41.9,196.0,...,False,3.436754,154.1,False,False,False,2.610070,4.976734,NaN,NaN
2,11028,25.0,150.0,NaN,2.500,1.29,NaN,35.8,62.7,133.0,...,False,0.570973,70.3,False,False,False,5.017280,3.605498,NaN,NaN


## Helpers

In [ ]:
def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def as_nullable_boolean(mask_defined: pd.Series, mask_true: pd.Series) -> pd.Series:
    out = pd.Series(np.nan, index=mask_defined.index, dtype="object")
    out.loc[mask_defined] = mask_true.loc[mask_defined].astype(bool)
    return out

def row_complete_for(df: pd.DataFrame, cols: List[str]) -> pd.Series:
    available_cols = [c for c in cols if c in df.columns]
    if not available_cols:
        return pd.Series(False, index=df.index)
    return df[available_cols].notna().all(axis=1)

def count_non_missing(s: pd.Series) -> int:
    return int(pd.Series(s).notna().sum())

def count_true(s: pd.Series) -> int:
    s2 = pd.Series(s)
    s2_non_missing = s2[s2.notna()]
    if len(s2_non_missing) == 0:
        return 0
    return int((s2_non_missing.astype(str).str.lower().isin(["true", "1"])).sum())

def pct_true(s: pd.Series) -> float:
    denom = count_non_missing(s)
    if denom == 0:
        return np.nan
    return round((count_true(s) / denom) * 100, 2)

## Thresholds from config

In [ ]:
# Endpoint thresholds
TG_HDL_CUTOFF = float(CFG["endpoints"]["primary"]["tg_hdl_cutoff"])
NON_HDL_CUTOFF = float(CFG["endpoints"]["secondary"]["non_hdl_high_mgdl"])
OGTT120_CUTOFF = float(CFG["endpoints"]["secondary"]["ogtt120_igt_mgdl"])

# Thyroid thresholds
TPO_ULN = float(CFG["tai_definition"]["A"]["tpo_uln"])
TSH_ULN = float(CFG["tai_definition"]["B"]["tsh_uln"])
TPO_HIGH_TITER = float(CFG["tai_definition"]["C"]["high_titer"])

# Euthyroid ranges
TSH_EU_LOW = float(CFG["thyroid"]["tsh_euthyroid_low"])
TSH_EU_HIGH = float(CFG["thyroid"]["tsh_euthyroid_high"])
FT4_REF_LOW = CFG["thyroid"].get("ft4_ref_low_optional", None)
FT4_REF_HIGH = CFG["thyroid"].get("ft4_ref_high_optional", None)

print("Thresholds:")
print(" TG/HDL cutoff:", TG_HDL_CUTOFF)
print(" non-HDL cutoff:", NON_HDL_CUTOFF)
print(" OGTT120 cutoff:", OGTT120_CUTOFF)
print(" anti-TPO ULN:", TPO_ULN, "| TSH ULN:", TSH_ULN, "| high-titer:", TPO_HIGH_TITER)
print(" Euthyroid TSH range:", (TSH_EU_LOW, TSH_EU_HIGH))
print(" FT4 ref optional:", (FT4_REF_LOW, FT4_REF_HIGH))

Thresholds:
 TG/HDL cutoff: 3.5
 non-HDL cutoff: 130.0
 OGTT120 cutoff: 140.0
 anti-TPO ULN: 34.0 | TSH ULN: 4.0 | high-titer: 100.0
 Euthyroid TSH range: (0.4, 4.0)
 FT4 ref optional: (None, None)


## Working copy and column sanity check

In [ ]:
df_a = df.copy()

required_base_cols = ["age", "anti_tpo", "tsh", "tg_hdl_ratio", "non_hdl"]
missing_required = [c for c in required_base_cols if c not in df_a.columns]
if missing_required:
    raise KeyError(f"Required columns missing in pcos_qc: {missing_required}")

anti_tpo = to_num(df_a["anti_tpo"])
tsh = to_num(df_a["tsh"])
tg_hdl = to_num(df_a["tg_hdl_ratio"])
non_hdl = to_num(df_a["non_hdl"])
glu120 = to_num(df_a["glu120"]) if "glu120" in df_a.columns else pd.Series(np.nan, index=df_a.index)
ft4 = to_num(df_a["ft4"]) if "ft4" in df_a.columns else pd.Series(np.nan, index=df_a.index)

## TAI exposure definitions

In [ ]:
# A: anti-TPO > ULN, only if anti-TPO measured
mask_A_defined = anti_tpo.notna()
mask_A_true = anti_tpo > TPO_ULN
df_a["tai_A"] = as_nullable_boolean(mask_A_defined, mask_A_true)

# B: anti-TPO > ULN AND TSH > ULN, only if BOTH measured
mask_B_defined = anti_tpo.notna() & tsh.notna()
mask_B_true = (anti_tpo > TPO_ULN) & (tsh > TSH_ULN)
df_a["tai_B"] = as_nullable_boolean(mask_B_defined, mask_B_true)

# C: anti-TPO > high-titer threshold, only if anti-TPO measured
mask_C_defined = anti_tpo.notna()
mask_C_true = anti_tpo > TPO_HIGH_TITER
df_a["tai_C"] = as_nullable_boolean(mask_C_defined, mask_C_true)

# Optional D/E
if "D_optional" in CFG["tai_definition"] and CFG["tai_definition"]["D_optional"].get("enabled", False):
    if "anti_tg" not in df_a.columns:
        logging.warning("D_optional enabled but anti_tg is missing. tai_D set to NA.")
        df_a["tai_D"] = np.nan
    else:
        anti_tg = to_num(df_a["anti_tg"])
        TG_ULN_D = float(CFG["tai_definition"]["D_optional"]["tg_uln"])
        mask_D_defined = anti_tpo.notna() | anti_tg.notna()
        mask_D_true = (anti_tpo > TPO_ULN) | (anti_tg > TG_ULN_D)
        df_a["tai_D"] = as_nullable_boolean(mask_D_defined, mask_D_true)

if "E_optional" in CFG["tai_definition"] and CFG["tai_definition"]["E_optional"].get("enabled", False):
    if "anti_tg" not in df_a.columns:
        logging.warning("E_optional enabled but anti_tg is missing. tai_E set to NA.")
        df_a["tai_E"] = np.nan
    else:
        anti_tg = to_num(df_a["anti_tg"])
        TG_ULN_E = float(CFG["tai_definition"]["E_optional"]["tg_uln"])
        mask_E_defined = anti_tpo.notna() & anti_tg.notna()
        mask_E_true = (anti_tpo > TPO_ULN) & (anti_tg > TG_ULN_E)
        df_a["tai_E"] = as_nullable_boolean(mask_E_defined, mask_E_true)

preview_cols = [c for c in ["anti_tpo", "tsh", "tai_A", "tai_B", "tai_C", "tai_D", "tai_E"] if c in df_a.columns]
df_a[preview_cols].head(5)

,anti_tpo,tsh,tai_A,tai_B,tai_C
0,13.8,0.969,False,False,False
1,12.6,2.050,False,False,False
2,150.0,2.500,True,False,True
3,9.0,1.200,False,False,False
4,9.0,1.050,False,False,False


## Endpoint definitions

In [ ]:
# Primary endpoint: TG/HDL > cutoff, only if tg_hdl_ratio defined
mask_ep_primary_defined = tg_hdl.notna()
mask_ep_primary_true = tg_hdl > TG_HDL_CUTOFF
df_a["ep_primary"] = as_nullable_boolean(mask_ep_primary_defined, mask_ep_primary_true)

# Secondary endpoint: non-HDL >= cutoff, only if non_hdl defined
mask_ep_non_hdl_defined = non_hdl.notna()
mask_ep_non_hdl_true = non_hdl >= NON_HDL_CUTOFF
df_a["ep_non_hdl"] = as_nullable_boolean(mask_ep_non_hdl_defined, mask_ep_non_hdl_true)

# Secondary endpoint: OGTT120 >= cutoff, only if glu120 defined
mask_ep_ogtt120_defined = glu120.notna()
mask_ep_ogtt120_true = glu120 >= OGTT120_CUTOFF
df_a["ep_ogtt120"] = as_nullable_boolean(mask_ep_ogtt120_defined, mask_ep_ogtt120_true)

df_a[[c for c in ["tg_hdl_ratio", "ep_primary", "non_hdl", "ep_non_hdl", "glu120", "ep_ogtt120"] if c in df_a.columns]].head(5)

,tg_hdl_ratio,ep_primary,non_hdl,ep_non_hdl,glu120,ep_ogtt120
0,2.049470,False,131.4,True,138.0,False
1,3.436754,False,154.1,True,137.0,False
2,0.570973,False,70.3,False,120.0,False
3,1.261618,False,88.9,False,147.0,True
4,3.820513,True,111.0,False,136.0,False


## Euthyroid flags

In [ ]:
# TSH-only euthyroid flag
mask_eu_tsh_defined = tsh.notna()
mask_eu_tsh_true = (tsh >= TSH_EU_LOW) & (tsh <= TSH_EU_HIGH)
df_a["euthyroid_tsh_only"] = as_nullable_boolean(mask_eu_tsh_defined, mask_eu_tsh_true)

# Strict euthyroid flag requires FT4 reference limits and FT4 measurement
if (FT4_REF_LOW is not None) and (FT4_REF_HIGH is not None) and ("ft4" in df_a.columns):
    mask_eu_strict_defined = tsh.notna() & ft4.notna()
    mask_eu_strict_true = (
        (tsh >= TSH_EU_LOW) & (tsh <= TSH_EU_HIGH) &
        (ft4 >= float(FT4_REF_LOW)) & (ft4 <= float(FT4_REF_HIGH))
    )
    df_a["euthyroid_strict"] = as_nullable_boolean(mask_eu_strict_defined, mask_eu_strict_true)
else:
    df_a["euthyroid_strict"] = np.nan

df_a[[c for c in ["tsh", "ft4", "euthyroid_tsh_only", "euthyroid_strict"] if c in df_a.columns]].head(5)

,tsh,ft4,euthyroid_tsh_only,euthyroid_strict
0,0.969,1.20,True,NaN
1,2.050,1.18,True,NaN
2,2.500,1.29,True,NaN
3,1.200,1.24,True,NaN
4,1.050,1.24,True,NaN


## Explicit analysis qualification flags

In [ ]:
eligibility_cfg = CFG.get("eligibility", {})

required_primary = eligibility_cfg.get("required_for_primary", [])
required_non_hdl = eligibility_cfg.get("required_for_secondary_non_hdl", [])
required_ogtt120 = eligibility_cfg.get("required_for_secondary_ogtt120", [])

# Minimal eligibility based on required variables from config
df_a["analysis_primary_minimal"] = row_complete_for(df_a, required_primary)
df_a["analysis_non_hdl_minimal"] = row_complete_for(df_a, required_non_hdl)
df_a["analysis_ogtt120_minimal"] = row_complete_for(df_a, required_ogtt120)

# Exposure definable using primary exposure A
df_a["exposure_tai_A_defined"] = df_a["tai_A"].notna()

# Endpoint definable flags
df_a["endpoint_primary_defined"] = df_a["ep_primary"].notna()
df_a["endpoint_non_hdl_defined"] = df_a["ep_non_hdl"].notna()
df_a["endpoint_ogtt120_defined"] = df_a["ep_ogtt120"].notna()

# Final analysis eligibility flags for downstream models
df_a["analysis_primary_eligible"] = df_a["exposure_tai_A_defined"] & df_a["endpoint_primary_defined"] & df_a["analysis_primary_minimal"]
df_a["analysis_non_hdl_eligible"] = df_a["exposure_tai_A_defined"] & df_a["endpoint_non_hdl_defined"] & df_a["analysis_non_hdl_minimal"]
df_a["analysis_ogtt120_eligible"] = df_a["exposure_tai_A_defined"] & df_a["endpoint_ogtt120_defined"] & df_a["analysis_ogtt120_minimal"]

df_a[[
    "analysis_primary_minimal",
    "analysis_non_hdl_minimal",
    "analysis_ogtt120_minimal",
    "exposure_tai_A_defined",
    "endpoint_primary_defined",
    "endpoint_non_hdl_defined",
    "endpoint_ogtt120_defined",
    "analysis_primary_eligible",
    "analysis_non_hdl_eligible",
    "analysis_ogtt120_eligible",
]].head(5)

,analysis_primary_minimal,analysis_non_hdl_minimal,analysis_ogtt120_minimal,exposure_tai_A_defined,endpoint_primary_defined,endpoint_non_hdl_defined,endpoint_ogtt120_defined,analysis_primary_eligible,analysis_non_hdl_eligible,analysis_ogtt120_eligible
0,True,True,True,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True,True,True,True


## Counts for exposures and endpoints

In [ ]:
rows = []

to_report = [
    "tai_A", "tai_B", "tai_C",
    "ep_primary", "ep_non_hdl", "ep_ogtt120",
    "euthyroid_tsh_only", "euthyroid_strict",
]

for opt in ["tai_D", "tai_E"]:
    if opt in df_a.columns:
        to_report.append(opt)

for var in to_report:
    s = df_a[var]
    rows.append({
        "variable": var,
        "n_non_missing": count_non_missing(s),
        "n_true": count_true(s),
        "pct_true_among_non_missing": pct_true(s),
    })

counts = pd.DataFrame(rows).sort_values("variable").reset_index(drop=True)

out_counts = tables_dir / "exposure_endpoint_counts.csv"
counts.to_csv(out_counts, index=False)

print("Saved counts table:", out_counts)
counts

Saved counts table: /content/outputs/tables/exposure_endpoint_counts.csv


,variable,n_non_missing,n_true,pct_true_among_non_missing
0,ep_non_hdl,1183,271,22.91
1,ep_ogtt120,1162,167,14.37
2,ep_primary,1183,82,6.93
3,euthyroid_strict,0,0,NaN
4,euthyroid_tsh_only,1182,1122,94.92
5,tai_A,1055,84,7.96
6,tai_B,1054,14,1.33
7,tai_C,1055,53,5.02


## Analysis flow / STROBE support

In [ ]:
analysis_flow = pd.DataFrame([
    {"stage": "n_total_rows", "n": int(len(df_a))},
    {"stage": "anti_tpo_measured", "n": int(anti_tpo.notna().sum())},
    {"stage": "tsh_measured", "n": int(tsh.notna().sum())},
    {"stage": "tai_A_defined", "n": int(df_a["tai_A"].notna().sum())},
    {"stage": "primary_endpoint_defined", "n": int(df_a["ep_primary"].notna().sum())},
    {"stage": "non_hdl_endpoint_defined", "n": int(df_a["ep_non_hdl"].notna().sum())},
    {"stage": "ogtt120_endpoint_defined", "n": int(df_a["ep_ogtt120"].notna().sum())},
    {"stage": "analysis_primary_minimal", "n": int(df_a["analysis_primary_minimal"].sum())},
    {"stage": "analysis_non_hdl_minimal", "n": int(df_a["analysis_non_hdl_minimal"].sum())},
    {"stage": "analysis_ogtt120_minimal", "n": int(df_a["analysis_ogtt120_minimal"].sum())},
    {"stage": "analysis_primary_eligible", "n": int(df_a["analysis_primary_eligible"].sum())},
    {"stage": "analysis_non_hdl_eligible", "n": int(df_a["analysis_non_hdl_eligible"].sum())},
    {"stage": "analysis_ogtt120_eligible", "n": int(df_a["analysis_ogtt120_eligible"].sum())},
])

analysis_flow_path = tables_dir / "analysis_flow_counts.csv"
analysis_flow.to_csv(analysis_flow_path, index=False)

print("Saved analysis flow table:", analysis_flow_path)
analysis_flow

Saved analysis flow table: /content/outputs/tables/analysis_flow_counts.csv


,stage,n
0,n_total_rows,1300
1,anti_tpo_measured,1055
2,tsh_measured,1182
3,tai_A_defined,1055
4,primary_endpoint_defined,1183
5,non_hdl_endpoint_defined,1183
6,ogtt120_endpoint_defined,1162
7,analysis_primary_minimal,1053
8,analysis_non_hdl_minimal,1053
9,analysis_ogtt120_minimal,1035


## Event counts by exposure for primary endpoint

In [ ]:
primary_event_by_taiA = pd.DataFrame()

mask_primary = df_a["analysis_primary_eligible"] == True
if mask_primary.any():
    tab = (
        df_a.loc[mask_primary]
        .groupby("tai_A", dropna=False)["ep_primary"]
        .agg(
            n="count",
            n_events=lambda s: int((pd.Series(s).astype(str).str.lower().isin(["true", "1"])).sum())
        )
        .reset_index()
    )
    tab["pct_events"] = np.where(tab["n"] > 0, (tab["n_events"] / tab["n"] * 100).round(2), np.nan)
    primary_event_by_taiA = tab

primary_event_path = tables_dir / "primary_event_counts_by_taiA.csv"
primary_event_by_taiA.to_csv(primary_event_path, index=False)

print("Saved primary event counts by tai_A:", primary_event_path)
primary_event_by_taiA

Saved primary event counts by tai_A: /content/outputs/tables/primary_event_counts_by_taiA.csv


,tai_A,n,n_events,pct_events
0,False,969,66,6.81
1,True,84,4,4.76


## Compact JSON summary

In [ ]:
summary = {
    "source_used": source_used,
    "n_rows": int(len(df_a)),
    "n_columns": int(df_a.shape[1]),
    "config_path": str(CONFIG_PATH),
    "thresholds": {
        "tg_hdl_cutoff": TG_HDL_CUTOFF,
        "non_hdl_cutoff": NON_HDL_CUTOFF,
        "ogtt120_cutoff": OGTT120_CUTOFF,
        "tpo_uln": TPO_ULN,
        "tsh_uln": TSH_ULN,
        "tpo_high_titer": TPO_HIGH_TITER,
        "tsh_euthyroid_low": TSH_EU_LOW,
        "tsh_euthyroid_high": TSH_EU_HIGH,
        "ft4_ref_low_optional": FT4_REF_LOW,
        "ft4_ref_high_optional": FT4_REF_HIGH,
    },
    "analysis_flow": analysis_flow.to_dict(orient="records"),
}

summary_path = reports_dir / "s3_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:", summary_path)
summary

Saved: /content/reports/s3_summary.json


{'source_used': '/content/pcos_qc.parquet',
 'n_rows': 1300,
 'n_columns': 77,
 'config_path': '/content/config.json',
 'thresholds': {'tg_hdl_cutoff': 3.5,
  'non_hdl_cutoff': 130.0,
  'ogtt120_cutoff': 140.0,
  'tpo_uln': 34.0,
  'tsh_uln': 4.0,
  'tpo_high_titer': 100.0,
  'tsh_euthyroid_low': 0.4,
  'tsh_euthyroid_high': 4.0,
  'ft4_ref_low_optional': None,
  'ft4_ref_high_optional': None},
 'analysis_flow': [{'stage': 'n_total_rows', 'n': 1300},
  {'stage': 'anti_tpo_measured', 'n': 1055},
  {'stage': 'tsh_measured', 'n': 1182},
  {'stage': 'tai_A_defined', 'n': 1055},
  {'stage': 'primary_endpoint_defined', 'n': 1183},
  {'stage': 'non_hdl_endpoint_defined', 'n': 1183},
  {'stage': 'ogtt120_endpoint_defined', 'n': 1162},
  {'stage': 'analysis_primary_minimal', 'n': 1053},
  {'stage': 'analysis_non_hdl_minimal', 'n': 1053},
  {'stage': 'analysis_ogtt120_minimal', 'n': 1035},
  {'stage': 'analysis_primary_eligible', 'n': 1053},
  {'stage': 'analysis_non_hdl_eligible', 'n': 1053},
 

## Analytical dataset recording

In [ ]:
out_parquet = intermediate_dir / "pcos_analysis.parquet"
out_csv = intermediate_dir / "pcos_analysis.csv"

try:
    df_a.to_parquet(out_parquet, index=False)
    print("Saved:", out_parquet)
except Exception as e:
    print("Parquet save failed. Reason:", repr(e))
    logging.warning("Parquet save failed: %r", e)

df_a.to_csv(out_csv, index=False)
print("Saved:", out_csv)

print("Final shape:", df_a.shape)

Saved: /content/outputs/intermediate/pcos_analysis.parquet
Saved: /content/outputs/intermediate/pcos_analysis.csv
Final shape: (1300, 77)


## Checklist

## S3 completion checklist

- [ ] Verify `exposure_endpoint_counts.csv` for reasonable prevalences.
- [ ] Verify `analysis_flow_counts.csv` for explicit cohort counts.
- [ ] Verify `primary_event_counts_by_taiA.csv` for event sparsity and event direction.
- [ ] Confirm `tai_A` is the primary exposure for Table 1 and primary models.
- [ ] Confirm `ep_primary` matches TG/HDL > cutoff.
- [ ] Proceed to S4 (descriptives / Table 1 / plots).